In [1]:
from pathlib import Path
import sys
import torch
import yaml

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.build import build_finetune_loader, build_finetune_model
from src.finetune.checkpoint import load_trainable_state
from src.finetune.loss import finetune_loss

CONFIG_PATH = ROOT / 'config/finetune_test.yaml'
LORA_PATH = None
BATCH_SIZE = 1
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = yaml.safe_load(CONFIG_PATH.read_text())
model_config = dict(config['model'])
model_config['path'] = str(ROOT / model_config['path'])
model_config['device'] = device
model = build_finetune_model(model_config)
if LORA_PATH is not None:
    checkpoint = torch.load(LORA_PATH, map_location='cpu', weights_only=False)
    load_trainable_state(model, checkpoint['model'])
model.train()

loader_config = dict(config['data']['train'])
loader_config['folders'] = [dict(folder, path=str(ROOT / folder['path'])) for folder in loader_config['folders']]
loader_config['batch_size'] = BATCH_SIZE
loader_config['num_workers'] = 0
loader = build_finetune_loader(
    loader_config,
    num_classes=config['model']['num_classes'],
    num_conditions=config['model']['num_conditions'],
    train=True,
)


In [2]:
batch = next(loader)
model_batch = {key: value.to(device) if torch.is_tensor(value) else value for key, value in batch.items()}
model.zero_grad(set_to_none=True)
with torch.autocast(device.type, enabled=device.type == 'cuda'):
    out = model(model_batch)
    loss, stats = finetune_loss(model_batch, out)

result = {
    'prompt': batch['prompt'][0]['type'],
    'cond': int(batch['cond'][0]),
    'mask_valid': bool(batch['mask_valid'][0]),
    'label_target': batch['label_target'][0].int().tolist(),
    'mask_shape': tuple(out['mask_logits'].shape),
    'total_tensor': float(loss.detach().cpu()),
    **stats,
}
result


{'prompt': 'mask',
 'cond': 1,
 'mask_valid': False,
 'label_target': [0, 0, 0],
 'mask_shape': (1, 1, 288, 288),
 'total_tensor': 0.2948324680328369,
 'class_loss_0': 0.2948324680328369,
 'class_acc_0': 1.0,
 'class_loss_1': 0.0,
 'class_acc_1': 0.0,
 'class_loss_2': 0.0,
 'class_acc_2': 0.0,
 'loss': 0.2948324680328369,
 'mask_bce': 0.0,
 'mask_dice': 0.0,
 'iou_loss': 0.0,
 'class_loss': 0.2948324680328369}